# Manga Title OCR with ROI Pre-processing

This notebook demonstrates a complete pipeline for manga title OCR using Region of Interest (ROI) pre-processing. The pipeline includes:

1. **Project Structure Setup** - Understanding the existing codebase
2. **ROI-based Preprocessing** - Detecting and cropping text-dense regions
3. **OCR Pipeline Integration** - Using preprocessed images for OCR
4. **Testing and Validation** - Verifying the pipeline works correctly

## 1. Project Structure Overview

The repository structure is as follows:

In [ ]:
# Directory structure
# manga_ocr/
# ├── ocr_manga_title/          # Main package
# │   ├── preprocess/          # Preprocessing module
# │   │   ├── steps/          # Individual preprocessing steps
# │   │   │   ├── roi.py      # ROI detection and cropping
# │   │   │   ├── grayscale.py
# │   │   │   ├── upscale.py
# │   │   │   ├── denoise.py
# │   │   │   └── binarize.py
# │   │   └── pipeline.py     # Preprocessing pipeline orchestration
# │   ├── models/             # OCR model implementations
# │   │   ├── tesseract_model.py
# │   │   ├── easyocr_model.py
# │   │   ├── paddle_model.py
# │   │   └── glm_ocr_model.py
# │   ├── schemas.py          # Pydantic data models
# │   └── engine.py           # Main OCR engine
# ├── tests/                  # Test suite
# ├── notebooks/              # Jupyter notebooks
# ├── preprocess.yaml         # Preprocessing configuration
# └── main.py                 # CLI entry point

## 2. ROI-based Preprocessing Implementation

The ROI (Region of Interest) preprocessing step detects text-dense regions in manga images and crops to those areas. This is crucial for manga because:
- Manga pages often have large margins and non-text areas
- Text is typically concentrated in specific regions
- Reducing the image area improves OCR accuracy and speed

In [ ]:
import cv2
import numpy as np

from ocr_manga_title.preprocess.steps.roi import ROIStep

# Initialize the ROI step
roi_step = ROIStep()
print(f"ROI Step Name: {roi_step.name}")
print(f"ROI Step Available: {roi_step.is_available}")

### How ROI Detection Works

The ROI step uses the following algorithm:

1. Convert to Grayscale - Simplify the image for processing
2. Adaptive Thresholding - Create a binary mask highlighting text regions
3. Contour Detection - Find all text-dense regions
4. Filter by Area - Keep only regions above min_area threshold
5. Merge Overlapping Rectangles - Combine nearby regions using IoU
6. Apply Padding - Expand the crop area with configurable padding
7. Coverage Check - Skip crop if region covers >95% of image

In [ ]:
# Create a sample manga-like image with text regions
def create_sample_manga_image():
    img = np.zeros((300, 400, 3), dtype=np.uint8)
    img[50:80, 50:200] = (200, 200, 200)
    img[120:150, 50:250] = (200, 200, 200)
    img[180:210, 50:350] = (200, 200, 200)
    noise = np.random.randint(0, 50, (300, 400, 3), dtype=np.uint8)
    img = cv2.add(img, noise)
    return img

sample_img = create_sample_manga_image()
print(f"Sample image shape: {sample_img.shape}")

In [ ]:
# Process with ROI step using default configuration
result_img, metadata = roi_step.process(sample_img, {})
print(f"Original shape: {sample_img.shape}")
print(f"Cropped shape: {result_img.shape}")
print(f"Metadata: {metadata}")

### ROI Configuration Options

The ROI step supports the following configuration parameters:

In [ ]:
# Example: Custom ROI configuration
custom_config = {
    "method": "contour",
    "min_area": 1000,
    "padding": 15,
    "merge_overlap": 0.2
}

result_img_custom, metadata_custom = roi_step.process(sample_img, custom_config)
print(f"Custom config result: {metadata_custom}")

## 3. Complete OCR Pipeline Integration

The preprocessing pipeline chains multiple steps together. The default order is:

In [ ]:
from ocr_manga_title.preprocess.pipeline import PreProcessingPipeline

pipeline = PreProcessingPipeline({}, "/tmp")
print('Pipeline steps:', pipeline.STEP_ORDER)

### Pipeline Configuration

Each preprocessing step can be individually enabled/disabled via configuration:

In [ ]:
from pathlib import Path

from ocr_manga_title.preprocess.pipeline import PreProcessingPipeline

test_img = np.random.randint(0, 255, (200, 200, 3), dtype=np.uint8)
test_img[50:150, 30:170] = (200, 200, 200)

import tempfile

temp_dir = Path(tempfile.mkdtemp())
img_path = temp_dir / "test_manga.png"
cv2.imwrite(str(img_path), test_img)
print(f"Test image saved to: {img_path}")

In [ ]:
preprocess_config = {
    'preprocessing': {
        'enabled': True,
        'debug': True,
        'roi': {'enabled': True, 'method': 'contour', 'min_area': 500, 'padding': 10},
        'grayscale': {'enabled': True},
        'upscale': {'enabled': True, 'method': 'cubic', 'scale_factor': 2},
        'denoise': {'enabled': True, 'method': 'gaussian', 'strength': 'light'},
        'binarize': {'enabled': True, 'method': 'otsu', 'block_size': 11, 'c': 2}
    }
}

pipeline = PreProcessingPipeline(preprocess_config, temp_dir)
result = pipeline.process(str(img_path))

print(f"Input: {result.input_path}")
print(f"Output: {result.output_path}")
print(f"Total time: {result.total_processing_time_ms}ms")

for step in result.steps:
    print(f"  - {step.step_name}: {step.success} (time: {step.processing_time_ms}ms)")

## 4. Testing and Validation

Let's verify the pipeline with different test scenarios:

In [ ]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt


def visualize_processing(original, processed, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
    axes[0].set_title('Original Image')
    axes[0].axis('off')

    axes[1].imshow(processed, cmap='gray')
    axes[1].set_title(title)
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

print('Testing ROI with text-like image...')
result_text, _ = roi_step.process(sample_img, {})
visualize_processing(sample_img, result_text, "ROI Cropped Image")
print(f"Text regions detected: {metadata['regions_detected']}")
print(f"Crop performed: {metadata['crop_performed']}")

In [ ]:
# Test with blank image (no text)
blank_img = np.zeros((200, 200), dtype=np.uint8)
result_blank, meta_blank = roi_step.process(blank_img, {})
print(f"Blank image regions detected: {meta_blank['regions_detected']}")
print(f"Blank image crop performed: {meta_blank['crop_performed']}")
print(f"Reason: {meta_blank.get('reason', 'N/A')}")

In [ ]:
# Test with small image (should skip)
small_img = np.random.randint(0, 255, (50, 50, 3), dtype=np.uint8)
result_small, meta_small = roi_step.process(small_img, {})
print(f"Small image crop performed: {meta_small['crop_performed']}")
print(f"Reason: {meta_small.get('reason', 'N/A')}")

In [ ]:
# Test with image where ROI covers >95% (should skip crop)
full_img = np.zeros((200, 200), dtype=np.uint8)
full_img[5:195, 5:195] = 255
result_full, meta_full = roi_step.process(full_img, {})
print(f"Full coverage image crop performed: {meta_full['crop_performed']}")
print(f"Coverage: {meta_full.get('coverage', 'N/A')}")
print(f"Reason: {meta_full.get('reason', 'N/A')}")

## 5. Integration with OCR Engine

The preprocessed image can be passed to the OCR engine for text extraction:

In [ ]:
from ocr_manga_title.config import load_config
from ocr_manga_title.engine import OCREngine

config = load_config()
ocr_config = load_ocr_config()
preprocess_config = load_preprocess_config()

engine = OCREngine(config, ocr_config, preprocess_config)

pipeline_result = engine.process(str(img_path))

print(f'OCR Results: {len(pipeline_result.ocr_results)} models processed')
if pipeline_result.extracted:
    print(f'Extracted Title: {pipeline_result.extracted.title_en or pipeline_result.extracted.title_ja}')
    print(f'Confidence: {pipeline_result.extracted.confidence}')
else:
    print('No title extracted')

## 6. Summary

This pipeline demonstrates:

- ROI Detection: Detects and crops text-dense regions
- Configurable Parameters: min_area, padding, merge_overlap
- Edge Case Handling: Small images, full coverage, no regions
- Pipeline Integration: Works with full OCR engine
- Debug Mode: Saves intermediate processing steps
- Validation: Tested with multiple test scenarios

# Clean up temporary files
import shutil
shutil.rmtree(temp_dir)